# x402 Batch-Settlement — Buyer / Facilitator Spike (Deno / TypeScript)

Phase 0 harness for the batch-settlement migration (see `assistent_plan.md` §F). It drives the
**batch-settlement** scheme against a locally-running facilitator, mirroring `x402_facilitator_demo_ts.ipynb`
(exact scheme). Use it to learn the client/facilitator API surface before wiring the scw_js server (Phase B).

**Prerequisites**
- Deno Jupyter kernel (same as the sibling `*_ts.ipynb` notebooks).
- The package's single **`x402_facilitator/.env`** (one level up — there is no per-notebook `.env`) with:
  `TEST_WALLET_PRIVATE_KEY` (a wallet funded on **Base Sepolia**: ETH for gas + testnet USDC via
  https://faucet.circle.com/), `NFT_WALLET_PUBLIC_KEY` (recipient), and `FACILITATOR_WALLET_PUBLIC_KEY`.
- A facilitator running locally: from the parent dir `npm install && npm run build && npm run dev` → `http://localhost:8080`.

**Why Base Sepolia:** the canonical `BATCH_SETTLEMENT_ADDRESS` (`0x4020…0003`) is deployed on Base Sepolia,
Optimism mainnet and Base mainnet — but **not** Optimism Sepolia. So testnet spikes use Base Sepolia.

> ⚠️ Cells that construct the deposit/voucher payload and hit `/settle` perform **real on-chain** actions
> and use batch-settlement client APIs that should be confirmed at runtime — they are the point of the spike.


In [ ]:
// Setup: imports + config
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount, generatePrivateKey } from "npm:viem@2/accounts";
import { createPublicClient, http, formatUnits } from "npm:viem@2";
import { baseSepolia, base, optimism } from "npm:viem@2/chains";

// All env lives in the package's single .env, one level up: x402_facilitator/.env
// (TEST_WALLET_PRIVATE_KEY, NFT_WALLET_PUBLIC_KEY, FACILITATOR_WALLET_*).
// `examplePath: null` disables dotenv's default assertion that every key in a
// ./.env.example must also be present — we're not using a per-notebook .env.
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
if (!PRIVATE_KEY) {
    throw new Error("TEST_WALLET_PRIVATE_KEY missing from x402_facilitator/.env — " +
        "add a key for a wallet funded on the selected network.");
}
const PAY_TO_ADDRESS = env.NFT_WALLET_PUBLIC_KEY ?? "0x209693Bc6afc0C5328bA36FaF03C514EF312287C";
const FACILITATOR_ADDRESS = env.FACILITATOR_WALLET_PUBLIC_KEY; // from ../.env — used for the gas check
const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);

// The receiver self-manages an authorizer key (signs claims/refunds). In production
// scw_js holds this and advertises its address in the 402's extra.receiverAuthorizer.
// For the spike we use a throwaway one.
const receiverAuthorizer = privateKeyToAccount(generatePrivateKey());

console.log("🚀 x402 Batch-Settlement Spike (Deno/TS)");
console.log(`   Payer (buyer)     : ${account.address}`);
console.log(`   Recipient         : ${PAY_TO_ADDRESS}`);
console.log(`   receiverAuthorizer: ${receiverAuthorizer.address}`);

## Network selection

Modeled on `x402_facilitator_demo.ipynb`, but **Optimism Sepolia is disabled** — the canonical
`BATCH_SETTLEMENT_ADDRESS` (`0x4020…0003`) is **not** deployed there. Valid networks (all have the contract):
Base Sepolia (default), Base Mainnet, Optimism Mainnet.

| Chain | Testnet | Mainnet |
|---|---|---|
| Base | `eip155:84532` ✅ | `eip155:8453` |
| Optimism | ~~`eip155:11155420`~~ (no contract) | `eip155:10` |


In [ ]:
// ── Network selection ──────────────────────────────────────────────
const USE_MAINNET = false;  // ⚠️ true = REAL MONEY. Keep false for the spike.
const USE_BASE = true;      // Base by default. (Optimism + testnet is unavailable.)

// Guard: Optimism Sepolia has no batch-settlement contract.
if (!USE_BASE && !USE_MAINNET) {
    throw new Error("Optimism Sepolia is unsupported for batch-settlement (no canonical contract). " +
        "Use Base Sepolia (USE_BASE=true) or a mainnet (USE_MAINNET=true).");
}

const NETWORK_CONFIG = {
    "base-testnet": {
        chain: baseSepolia, chainId: 84532, caip2Network: "eip155:84532" as const,
        networkName: "Base Sepolia (Testnet)", usdcName: "USDC",
        usdcAddress: "0x036CbD53842c5426634e7929541eC2318f3dCF7e" as `0x${string}`,
        rpcUrl: "https://sepolia.base.org", explorer: "https://sepolia.basescan.org",
        faucet: "https://faucet.circle.com/",
    },
    "base-mainnet": {
        chain: base, chainId: 8453, caip2Network: "eip155:8453" as const,
        networkName: "Base Mainnet", usdcName: "USD Coin",
        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913" as `0x${string}`,
        rpcUrl: "https://mainnet.base.org", explorer: "https://basescan.org",
        faucet: "Bridge: https://bridge.base.org",
    },
    "optimism-mainnet": {
        chain: optimism, chainId: 10, caip2Network: "eip155:10" as const,
        networkName: "Optimism Mainnet", usdcName: "USD Coin",
        usdcAddress: "0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85" as `0x${string}`,
        rpcUrl: "https://mainnet.optimism.io", explorer: "https://optimistic.etherscan.io",
        faucet: "Bridge: https://app.optimism.io/bridge",
    },
};
const configKey = USE_MAINNET ? (USE_BASE ? "base-mainnet" : "optimism-mainnet") : "base-testnet";
const config = NETWORK_CONFIG[configKey];

// Max per-request price (6-decimal USDC). Channel deposit ≈ depositMultiplier(5) × this.
const MAX_PRICE = "4000"; // $0.004  → deposit ≈ $0.02

const FACILITATOR_URL = "http://localhost:8080"; // or "https://facilitator.fretchen.eu"
const VERIFY_URL = `${FACILITATOR_URL}/verify`;
const SETTLE_URL = `${FACILITATOR_URL}/settle`;
const SUPPORTED_URL = `${FACILITATOR_URL}/supported`;

console.log(USE_MAINNET ? `🚨 REAL MONEY on ${config.networkName}` : `🧪 ${config.networkName}`);
console.log(`   ${config.caip2Network}  •  USDC ${config.usdcName} @ ${config.usdcAddress}`);
console.log(`   Facilitator: ${FACILITATOR_URL}`);

## 1. `/supported` — assert the facilitator advertises `batch-settlement`

This is the core Phase A check: after registering `BatchSettlementEvmScheme` in `facilitator_instance.ts`,
the facilitator's `/supported` should list the `batch-settlement` scheme for each network.


In [ ]:
const supported = await (await fetch(SUPPORTED_URL)).json();
const kinds = supported.kinds ?? [];
const batchKinds = kinds.filter((k: any) => k.scheme === "batch-settlement");
const exactKinds = kinds.filter((k: any) => k.scheme === "exact");

console.log(`schemes advertised: exact=${exactKinds.length}, batch-settlement=${batchKinds.length}`);
console.log("batch-settlement kinds:", JSON.stringify(batchKinds, null, 2));

if (batchKinds.some((k: any) => k.network === config.caip2Network)) {
    console.log(`✅ batch-settlement advertised for ${config.caip2Network}`);
} else {
    console.log(`❌ batch-settlement NOT advertised for ${config.caip2Network} — check facilitator registration`);
}
// Note: with no receiverAuthorizer configured (self-managed receiver), `extra.receiverAuthorizer`
// should be absent here. Confirm — this is spike-unknown #1.

## 2. Batch-settlement client setup

Unlike the exact scheme there is **no `registerBatchSettlementEvmScheme` helper** — construct the scheme
and register it manually on the `x402Client`. A browser client would back `storage` with `localStorage`;
here we use the in-memory storage.


In [ ]:
import { x402Client } from "npm:@x402/fetch@^2.17.0";
import { BatchSettlementEvmScheme, InMemoryClientChannelStorage }
    from "npm:@x402/evm@^2.17.0/batch-settlement/client";

const storage = new InMemoryClientChannelStorage();
const scheme = new BatchSettlementEvmScheme(
    { address: account.address, signTypedData: (a: any) => account.signTypedData(a) },
    { storage },
);

const client = new x402Client();
client.register(config.caip2Network, scheme);
console.log("✅ batch-settlement client registered for", config.caip2Network);

## 3. Build batch-settlement payment requirements

In production these come from the **server's 402 response** (the scw_js resource server injects these via the
server scheme's `enhancePaymentRequirements`). **Confirmed required in `extra`:** `receiverAuthorizer`
(non-zero — the client throws `Payment requirements must include a non-zero extra.receiverAuthorizer` without it)
and `withdrawDelay`, plus the usual EIP-712 `name`/`version`.


In [ ]:
const paymentRequirements = {
    scheme: "batch-settlement",
    network: config.caip2Network,
    amount: MAX_PRICE,
    asset: config.usdcAddress,
    payTo: PAY_TO_ADDRESS,
    maxTimeoutSeconds: 3600,
    extra: {
        name: config.usdcName,
        version: "2",
        receiverAuthorizer: receiverAuthorizer.address, // server's self-managed authorizer in prod
        withdrawDelay: 86400,
    },
};

const paymentRequired = {
    x402Version: 2,
    accepts: [paymentRequirements],
    resource: { url: "https://example.com/llm", description: "batch-settlement spike", mimeType: "application/json" },
    extensions: {},
};
console.log(JSON.stringify(paymentRequirements, null, 2));

## 4. Create the deposit + first cumulative voucher payload

On the first request the client opens a channel: an EIP-3009 deposit into the batch-settlement contract plus
an initial cumulative voucher. `createPaymentPayload` handles the signing. **Confirmed payload shape:**
`payload.type == "deposit"` with `channelConfig` (payer, payerAuthorizer, receiver, receiverAuthorizer, token,
withdrawDelay, salt), `voucher` (channelId, maxClaimableAmount, signature), and `deposit`. The deposit is sized
by the deposit policy (default `depositMultiplier: 5`), so the payer needs ≥ 5×MAX_PRICE USDC to settle.


In [ ]:
const paymentPayload = await client.createPaymentPayload(paymentRequired as any);
console.log("✅ payload created (deposit + voucher):");
console.log(JSON.stringify(paymentPayload, null, 2).slice(0, 1200));

## 5. `/verify` the payload


In [ ]:
const verifyRes = await fetch(VERIFY_URL, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ paymentPayload, paymentRequirements }),
});
const verifyResult = await verifyRes.json();
console.log(`verify (${verifyRes.status}):`, JSON.stringify(verifyResult, null, 2));

## 6. Pre-settlement balance check (like the demo notebook)

Settlement escrows the deposit on-chain, so the **payer needs USDC** (≈ `depositMultiplier` × MAX_PRICE) and
the **facilitator wallet needs ETH** for gas. Check before spending.


In [ ]:
const publicClient = createPublicClient({ chain: config.chain, transport: http(config.rpcUrl) });
const erc20 = [{ inputs: [{ name: "a", type: "address" }], name: "balanceOf",
    outputs: [{ name: "", type: "uint256" }], stateMutability: "view", type: "function" }] as const;

const payerUsdc = await publicClient.readContract({
    address: config.usdcAddress, abi: erc20, functionName: "balanceOf", args: [account.address] });
const DEPOSIT_MULTIPLIER = 5n; // batch-settlement default
const needed = BigInt(MAX_PRICE) * DEPOSIT_MULTIPLIER;

console.log(`💵 Payer USDC: ${formatUnits(payerUsdc, 6)}  (deposit needs ≈ ${formatUnits(needed, 6)})`);
if (payerUsdc < needed) {
    console.log(`   ⚠️ insufficient USDC on ${config.networkName} — ${config.faucet}`);
} else {
    console.log(`   ✅ enough USDC to open the channel`);
}

// The facilitator submits the deposit tx, so it needs ETH for gas (from ../.env).
if (FACILITATOR_ADDRESS) {
    const facEth = await publicClient.getBalance({ address: FACILITATOR_ADDRESS as `0x${string}` });
    console.log(`⛽ Facilitator ETH: ${formatUnits(facEth, 18)}  (${FACILITATOR_ADDRESS})`);
    if (facEth === 0n) console.log(`   ⚠️ facilitator has no ETH for gas on ${config.networkName}`);
} else {
    console.log("⛽ FACILITATOR_WALLET_PUBLIC_KEY not found in ../.env — skipping gas check");
}

## 7. `/settle` — submit the deposit on-chain

> ⚠️ Real on-chain tx: escrows USDC into the batch-settlement contract. The facilitator dispatches the deposit
> internally by payload type — no separate endpoint. batch-settlement is **fee-free** (no fee `transferFrom`).


In [ ]:
if (USE_MAINNET) throw new Error("Refusing to auto-settle on mainnet. Set USE_MAINNET=false or run this cell deliberately.");

const settleRes = await fetch(SETTLE_URL, {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ paymentPayload, paymentRequirements }),
});
const settleResult = await settleRes.json();
console.log(`settle (${settleRes.status}):`, JSON.stringify(settleResult, null, 2));
if (settleResult.transaction) console.log(`🔗 ${config.explorer}/tx/${settleResult.transaction}`);
// Fee-free: settleResult should carry NO facilitatorFees / fee.collected.

## Spike findings (confirmed) + what feeds Phase B

- ✅ **Facilitator advertises batch-settlement** for all networks via `/supported`, with **no** `receiverAuthorizer`
  in `extra` → confirms self-managed receiver (scw_js holds the authorizer key, not the facilitator).
- ✅ **Requirements need `extra.receiverAuthorizer` (non-zero) + `withdrawDelay`** — the client throws without them.
- ✅ **Deposit payload shape**: `type="deposit"` + channelConfig + voucher (channelId/maxClaimableAmount/signature) + deposit.
- ✅ **Verify wiring works**: an unfunded payer returns `invalid_batch_settlement_evm_insufficient_balance` — i.e.
  signature/domain/channel config are all accepted, only the balance stops it. (Automated: see
  `test/integration/x402_batch_settlement.integration.test.js`, run via `npm run test:integration`.)

**Still to exercise here (needs a funded Base Sepolia payer):** `/settle` (submits the deposit on-chain), then the
claim/settle/refund lifecycle. Deposit = `depositMultiplier` (5) × MAX_PRICE, so fund the payer accordingly.

**Phase B next:** wire the scw_js server scheme (`BatchSettlementEvmScheme(receiver, { storage: s3Storage })`) + the
S3 `ChannelStorage`, use `setSettlementOverrides` to claim the actual post-generation cost, then the claim/settle
cron. See `assistent_plan.md` §F.
